In [2]:
import pandas as pd
import numpy as np

import sys
sys.path.append('../src/')
import msu_labels as msu
%load_ext autoreload
%autoreload 2

In [4]:
msu.get_msu_projects()

['arcos',
 'birdlife',
 'cerath',
 'dnrc',
 'drek',
 'epz',
 'fairtree',
 'gosh',
 'herp',
 'icraf',
 'iggs',
 'ileg',
 'inec',
 'itf',
 'kev',
 'kft',
 'mkec',
 'pado',
 'recor',
 'sadhana',
 'safi',
 'wag',
 'wfz']

In [6]:
df = msu.combine_projects(outfile='../data/msu_field/msu_comb.shp')

/Users/jessica.ertel/github/dinov3_species/notebooks/../src/msu_labels.py:127: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = gpd.GeoDataFrame(pd.concat(combined_frames, ignore_index=True))
/Users/jessica.ertel/miniforge3/envs/dtree/lib/python3.12/site-packages/pyogrio/raw.py:709: RuntimeWarning: Field date create as date field, though DateTime requested.
  ogr_write(


In [7]:
df.head()

,sid,pid,date,treeid,lat_t,long_t,species,cluster,dbh__cm_,crown_d_ma,crown_d_90,height,remarks,geometry,project
0,sadhana,11.0,2023-06-21,1.0,0.880889,36.834244,acacia,None,36.0,None,None,None,None,POINT (4100369.335 98063.966),sadhana
1,sadhana,11.0,2023-06-21,2.0,0.880839,36.834339,Croton megalocarpus,None,10.1,None,None,None,None,POINT (4100379.849 98058.399),sadhana
2,sadhana,11.0,2023-06-21,3.0,0.880761,36.834017,ziziphus,None,58.7,None,None,None,None,POINT (4100343.979 98049.74),sadhana
3,sadhana,11.0,2023-06-21,4.0,0.880636,36.834253,ziziphus,None,16.9,None,None,None,None,POINT (4100370.263 98035.823),sadhana
4,sadhana,11.0,2023-06-21,5.0,0.880542,36.834289,None,None,15.8,None,None,None,None,POINT (4100374.283 98025.309),sadhana


In [32]:
df.sid.value_counts()

sid
WFZ           486
MKEC          382
cerath        359
safi          328
EPZ           316
WAG           296
Arcos RW      280
TURKANA       265
ILEG          211
DNRC          208
ITF           184
ICRAF 2_RW    173
KEV           147
ICRAF 1_RW    141
Gosh          127
sadhana        95
DREK           93
BirdLife       72
pado           69
Herp           67
fairtree       62
RECOR RW       52
inec           25
IGGS            7
KFT             1
Name: count, dtype: int64

In [8]:
df.species.value_counts()

species
Vitellaria paradoxa     487
Mangifera indica        367
Acacia nilotica         257
Grevillea robusta       204
Persea americana        152
                       ... 
Euphorbia umbellata       1
Balenites aegyptiaca      1
African olea              1
Jacaranda                 1
A                         1
Name: count, Length: 435, dtype: int64

In [10]:
df.species.unique()

array(['acacia', 'Croton megalocarpus', 'ziziphus', None, 'Jacaranda',
       'Citrus aurantiifolia', 'baelentyes', 'African olea',
       'Azadirachta indica', 'plumeria', 'Balenites aegyptiaca',
       'Euphorbia umbellata', 'Grevillea robusta', 'Ficus thoningii',
       'Albizia lebbeck', 'Euphorbia tirucalli', 'Mangifera indica',
       'Sena spectabillis', 'Grewia bicolor', 'Persea americana',
       'Eucalyptus saligna', 'Markhamia lutea', 'Sena sepectabilis',
       'Ficus bussei', 'Ozoroa leticulata', 'Combretum collinum',
       'Combretum molle', 'Jacaranda mimosifolia', 'Euphorbia ingens',
       'Acacia sieberiana', 'Erythrina abyssinica', 'Myrica solicitolia',
       'Ozoroa insgins', 'Acacia hockii', 'Rhus natalensis',
       'Senna spectabilis', 'Acacia ayssinica', 'Ximenia americana',
       'Rhamnus cathartica', 'Ephorbia ingens', 'Myrica solicifolia',
       'Lannea discolor', 'Persea americanap', 'Ficus exasperata',
       'Morinda lucida', 'Petersianthus africanus',

## Species Cleaning

In [12]:
dfn = msu.normalize_species_column(
    og_df,
    species_col="species",
    # genus_initial_map={  # tweak as you see these in your data
    #     "p": "persea",     # handles 'p. americana'
    #     "g": "grevillea",  # handles 'g.robusta' -> 'grevillea robusta' if present
    #     "m": "mangifera",
    #     "c": "cordia",
    # },
    known_fixes={
        "grevillea robutsa": "grevillea robusta",
        "mangnifera indica": "mangifera indica",
        "mangnigera indica": "mangifera indica",
        "ficus thoningii": "ficus thonningii",
        "fiscus sur": "ficus sur",
        "fucus spp": "ficus spp",
        "markhemia lutea": "markhamia lutea",
        "sena spectabillis": "senna spectabilis",
        "sena sepectabilis": "senna spectabilis",
        "teminalia superba": "terminalia superba",
        "terminallia sp": "terminalia spp",
    }
)

In [14]:
# 2) (Optional) Flag multi-species entries for review/exploding later
dfn["species_is_multi"] = dfn["species_norm"].str.contains(r",|;|/| and ", na=False)

# 3) Get suggestions for merges (keeps it simple, no hyphens, genus-bucketed)
merge_suggestions = msu.suggest_species_merges(dfn["species_norm"], min_similarity=0.9, by_genus=True, min_count=1)


In [17]:
og_df, report = msu.species_eda(df)
report

{'nan_rows': 329,
 'none_rows': 329,
 'blank_rows': 0,
 'dropped_rows_total': 329,
 'rows_after_clean': 4117,
 'unique_species_count': 411}

In [19]:
dfn['species_norm'].nunique(dropna=True)

399

In [50]:
df['date'].agg(['min','max'])

min   2023-01-11
max   2026-06-26
Name: date, dtype: datetime64[ns]

In [52]:
df.date.nlargest(2)

415   2026-06-26
411   2025-06-26
Name: date, dtype: datetime64[ns]